# leodragkit aero tutorial: building and plotting a spacecraft

A running example that grows with each aero task: every task adds a section showing what its code produces. Tests check the numbers; this notebook shows the result.

**Before running.** Activate the `leodragkit` environment and install the optional plotting library: `conda install -c conda-forge plotly`. The figures are interactive: drag to rotate, scroll to zoom, click a legend entry to hide it.

**In git.** Clear all outputs before committing (in Jupyter or VS Code: *Clear All Outputs*), so the repository holds code and text, not large figures.

| Section | Task | What it shows |
| --- | --- | --- |
| 1 | A3a | one triangle: corner order sets the outward normal |
| 2 | A3a | a cube built by hand as a `PanelSet` |
| 3 | A3p | the cube in 3-D, coloured by component and by material |
| 4 | A3a, A3p | a macro model: panels without positions |


In [ ]:
%load_ext autoreload
%autoreload 2

import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
import numpy as np

from leodragkit.aero.geometry import (
    compute_triangle_properties,
    panel_set_from_panels,
    panel_set_from_triangles,
)
from leodragkit.plots.spacecraft import plot_panel_set_3d

## 1. One triangle (A3a)

A triangle's outward normal follows its corner order: counter-clockwise when seen from outside. These corners go counter-clockwise seen from +z, so the normal points along +z. The area is half the length of the cross product of two edges.

In [3]:
triangle_m = np.array([[[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]])
properties = compute_triangle_properties(triangle_m)
print("normal:  ", properties.outward_normals[0])
print("area:    ", properties.areas_m2[0], "m²")
print("centroid:", properties.centroids_m[0], "m")

normal:   [0. 0. 1.]
area:     0.5 m²
centroid: [0.33333333 0.33333333 0.        ] m


Listing the same corners in the opposite order flips the normal: the panel now faces −z.

In [4]:
reversed_properties = compute_triangle_properties(triangle_m[:, ::-1, :])
print("normal with the corners reversed:", reversed_properties.outward_normals[0])

normal with the corners reversed: [ 0.  0. -1.]


## 2. A cube built by hand (A3a)

A unit cube centred on the body origin: six faces, two triangles each, every face counter-clockwise seen from outside. Task A3b will replace this by-hand construction with primitives (`box`, `plate`, `cylinder`); writing it out once shows what those primitives produce.

The four side faces are the component `body` and the two z faces are `end_caps`; the ±y faces are `solar_cell` material and the rest `aluminium`.

In [5]:
corners_m = np.array([[x, y, z] for x in (-0.5, 0.5) for y in (-0.5, 0.5) for z in (-0.5, 0.5)])
faces = [
    (0, 1, 3, 2),
    (4, 6, 7, 5),
    (0, 4, 5, 1),
    (2, 3, 7, 6),
    (0, 2, 6, 4),
    (1, 5, 7, 3),
]  # −x +x −y +y −z +z
cube_triangles_m = np.array(
    [corners_m[list(triangle)] for a, b, c, d in faces for triangle in ((a, b, c), (a, c, d))]
)

cube = panel_set_from_triangles(
    cube_triangles_m,
    component_ids=[0] * 8 + [1] * 4,
    material_ids=[0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0],
    component_names=("body", "end_caps"),
    material_names=("aluminium", "solar_cell"),
    source="unit cube, built by hand",
)
print(cube.panel_count, "panels; total area", cube.areas_m2.sum(), "m²")
print("has positions:", cube.has_positions, "| has triangles:", cube.has_triangles)

12 panels; total area 6.0 m²
has positions: True | has triangles: True


## 3. The cube in 3-D (A3p)

Each component is one trace, so the legend lists them. The lines are the outward normals, one per triangle, starting at its centroid; the red, green and blue lines are the body x, y and z axes. Every normal should point away from the cube: one pointing inward would mean a triangle with its corners in the wrong order.

In [28]:
figure = plot_panel_set_3d(cube)
figure

The same cube coloured by material: the ±y faces are solar cells.

In [ ]:
plot_panel_set_3d(cube, color_by="material")

## 4. A macro model: panels without positions (A3a, A3p)

Published macro models list each panel's outward normal, area and material, often with no position. Here is a simple one for a 3U CubeSat (0.1 × 0.1 × 0.34 m) with one deployed solar panel (0.3 × 0.1 m), listed once for each side. The normals are rounded to four decimals, as tables often are; `panel_set_from_panels` normalises them.

With no positions there is no shape to draw. The plot shows each normal from the origin, with length proportional to the panel's area, and the title says so. A set like this supports forces, but not shadowing or moments.

In [ ]:
normals = [
    [1, 0, 0],
    [-1, 0, 0],
    [0, 1, 0],
    [0, -1, 0],
    [0, 0, 1],
    [0, 0, -1],  # bus faces
    [0.7071, 0.7071, 0],
    [-0.7071, -0.7071, 0],  # one deployed panel, front and back, rounded
]
areas_m2 = [0.034, 0.034, 0.034, 0.034, 0.01, 0.01, 0.03, 0.03]
macro_model = panel_set_from_panels(
    normals,
    areas_m2,
    component_ids=[0] * 6 + [1] * 2,
    material_ids=[0] * 6 + [1] * 2,
    component_names=("bus", "deployed_panel"),
    material_names=("aluminium", "solar_cell"),
    source="3U CubeSat macro model (illustrative)",
)
plot_panel_set_3d(macro_model)

## Saving a figure

Plot functions never write files. To keep a figure, save it yourself, for example into the output directory from your main config:

```python
figure.write_html("cube.html")
```

## Coming next

- **A3b**: primitives (`box`, `plate`, `cylinder`) and poses, replacing the by-hand cube with a 3U CubeSat and its deployed panels.
- **A3c**: the same spacecraft read from a YAML description.
- **A3d**: macro-model tables and STL or OBJ meshes from files.
- **A4**: the incoming flow drawn on the spacecraft (plot level 2).